# Notebook 05 — GraphSAGE Training

## Dynamic Heterogeneous Graph Neural Network for Bank Marketing Prediction using GraphSAGE

### Objective

Train the first heterogeneous GraphSAGE model using the **verified graph produced by Notebook 04**.

This notebook is intentionally the end of the initial build phase.

```text
Notebook 04
HeteroData
    ↓
Heterogeneous GraphSAGE
    ↓
Customer-node embeddings
    ↓
Binary classifier
    ↓
Validation metrics
    ↓
Best checkpoint
```

### Strict training boundary

This notebook:

- trains only on `customer.train_mask`
- uses `customer.val_mask` for model selection and early stopping
- does **not** use `customer.test_mask` for tuning or checkpoint selection
- reports validation metrics during training
- saves the best validation checkpoint
- saves training history
- does **not** perform final test evaluation
- does **not** freeze the final model
- does **not** start tuning experiments

After this notebook, the project intentionally stops at the model-training checkpoint.

The user will inspect the validation results and decide whether further experimentation is required.


# 1. Verified Graph Contract

Notebook 04 successfully reloaded the graph with:

```text
customer
job
education
marital
contact
month
```

and customer-to-entity / reverse relations.

The verified customer feature matrix is:

```text
customer.x = [45,211, 50]
```

The graph also contains:

```text
customer.y
customer.train_mask
customer.val_mask
customer.test_mask
```

### Important

Notebook 05 uses the **actual saved graph artifact and its actual edge types**.

It does not reconstruct the graph or silently add new relations.


# 2. Model Architecture

The model uses a compact heterogeneous GraphSAGE architecture.

```text
HeteroData
   │
   ├── customer features
   ├── entity features
   └── typed relations
          ↓
   Heterogeneous SAGE Layer 1
          ↓
       ReLU
          ↓
       Dropout
          ↓
   Heterogeneous SAGE Layer 2
          ↓
       ReLU
          ↓
       Dropout
          ↓
   Customer embedding
          ↓
   Linear classifier
          ↓
      logit
          ↓
     sigmoid
          ↓
  subscription probability
```

`SAGEConv` is used separately for each heterogeneous relation through `HeteroConv`.

Only the customer-node representation is passed to the final binary classifier.


In [ ]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

from pathlib import Path
import json
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from IPython.display import display

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

HIDDEN_DIM = 64
DROPOUT = 0.30
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 150
PATIENCE = 20

# PR-AUC is the primary validation selection metric because
# the positive class is substantially smaller than the negative class.
PRIMARY_METRIC = "val_pr_auc"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GRAPH_PATH = PROJECT_ROOT / "artifacts" / "graph" / "bank_heterodata.pt"
GRAPH_METADATA_PATH = PROJECT_ROOT / "artifacts" / "graph" / "graph_metadata.json"

MODELS_DIR = PROJECT_ROOT / "artifacts" / "models"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = MODELS_DIR / "graphsage_best_checkpoint.pt"
HISTORY_PATH = RESULTS_DIR / "graphsage_training_history.json"
CURVE_PATH = RESULTS_DIR / "graphsage_training_curves.png"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project root:", PROJECT_ROOT)
print("Graph:", GRAPH_PATH)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


# 3. Verify Required Graph Artifact

Notebook 04 is a hard dependency.

If the graph artifact is missing, training must stop rather than reconstructing or fabricating a graph.


In [ ]:
# ============================================================
# 2. Verify Graph Artifact
# ============================================================

if not GRAPH_PATH.exists():
    raise FileNotFoundError(
        f"Graph artifact not found: {GRAPH_PATH}. "
        "Run and verify Notebook 04 first."
    )

if not GRAPH_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Graph metadata not found: {GRAPH_METADATA_PATH}. "
        "Run and verify Notebook 04 first."
    )

print("Required graph artifacts found.")


# 4. Load the Saved Heterogeneous Graph

PyTorch 2.6+ may default to restricted `weights_only=True` loading.

The artifact is intentionally a PyTorch Geometric `HeteroData` object, so the notebook explicitly requests normal object loading where supported.


In [ ]:
# ============================================================
# 3. Load HeteroData
# ============================================================

try:
    data = torch.load(
        GRAPH_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    data = torch.load(
        GRAPH_PATH,
        map_location="cpu"
    )

if not isinstance(data, HeteroData):
    raise TypeError(
        f"Expected HeteroData, received {type(data)}"
    )

with open(GRAPH_METADATA_PATH, "r", encoding="utf-8") as f:
    graph_metadata = json.load(f)

print(data)
print("\nNode types:", data.node_types)
print("Edge types:", data.edge_types)


# 5. Validate the Training Contract

Before creating the model, verify:

- customer features exist
- customer labels exist
- train/validation/test masks exist
- masks are mutually exclusive and exhaustive
- the graph contains the expected verified node types
- entity features are present
- every relation has an `edge_index`


In [ ]:
# ============================================================
# 4. Graph Training Contract
# ============================================================

required_customer_fields = [
    "x",
    "y",
    "train_mask",
    "val_mask",
    "test_mask",
]

for field in required_customer_fields:
    if field not in data["customer"]:
        raise KeyError(
            f"Missing customer graph field: {field}"
        )

expected_node_types = {
    "customer",
    "job",
    "education",
    "marital",
    "contact",
    "month",
}

actual_node_types = set(data.node_types)

if actual_node_types != expected_node_types:
    raise ValueError(
        "The saved graph node types differ from the verified Notebook 04 graph.\n"
        f"Expected: {sorted(expected_node_types)}\n"
        f"Actual:   {sorted(actual_node_types)}"
    )

num_customers = data["customer"].num_nodes

assert data["customer"].x.shape[0] == num_customers
assert data["customer"].y.shape[0] == num_customers

train_mask = data["customer"].train_mask.bool()
val_mask = data["customer"].val_mask.bool()
test_mask = data["customer"].test_mask.bool()

assert train_mask.shape[0] == num_customers
assert val_mask.shape[0] == num_customers
assert test_mask.shape[0] == num_customers

mask_sum = (
    train_mask.to(torch.int8)
    + val_mask.to(torch.int8)
    + test_mask.to(torch.int8)
)

assert torch.all(mask_sum == 1)

for edge_type in data.edge_types:
    assert "edge_index" in data[edge_type]

print("Training contract passed.")
print("Customers:", num_customers)
print("Customer feature dimension:", data["customer"].x.shape[1])
print("Train customers:", int(train_mask.sum()))
print("Validation customers:", int(val_mask.sum()))
print("Test customers:", int(test_mask.sum()))


# 6. Prepare Target and Class Weight

The target is:

```text
no  → 0
yes → 1
```

Because the positive class is only about 11.7% of the population, the initial training loss uses:

```text
BCEWithLogitsLoss(pos_weight=negative_count / positive_count)
```

The weight is computed **from the training mask only**.

This is not test-set tuning.


In [ ]:
# ============================================================
# 5. Target and Training Class Weight
# ============================================================

y = data["customer"].y.long()

train_y = y[train_mask]

negative_count = int((train_y == 0).sum())
positive_count = int((train_y == 1).sum())

if positive_count == 0:
    raise ValueError("Training set contains no positive examples.")

pos_weight_value = negative_count / positive_count

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE
)

print("Training target distribution:")
print(f"Negative (0): {negative_count:,}")
print(f"Positive (1): {positive_count:,}")
print(f"pos_weight  : {pos_weight_value:.4f}")


# 7. Move Graph to the Training Device

This is a full-graph training setup.

The complete heterogeneous graph is loaded into memory, while the loss is calculated only on the customer nodes selected by the appropriate mask.

This keeps the first GraphSAGE implementation understandable and reproducible.


In [ ]:
# ============================================================
# 6. Move Graph to Device
# ============================================================

data = data.to(DEVICE)

train_mask = data["customer"].train_mask.bool()
val_mask = data["customer"].val_mask.bool()
test_mask = data["customer"].test_mask.bool()

print("Graph moved to:", DEVICE)


# 8. Define Heterogeneous GraphSAGE

`HeteroConv` applies a separate `SAGEConv` operator for every relation type.

For each relation:

```text
source node features
        ↓
SAGEConv
        ↓
hidden representation
```

The outputs from different relation types are aggregated.

The model keeps the implementation intentionally compact so that its behavior is easy to inspect before any tuning experiments.


In [ ]:
# ============================================================
# 7. Heterogeneous GraphSAGE Model
# ============================================================

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self,
        metadata,
        hidden_dim=64,
        dropout=0.30
    ):
        super().__init__()

        node_types, edge_types = metadata

        self.dropout = dropout

        self.conv1 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim,
                    aggr="mean"
                )
                for edge_type in edge_types
            },
            aggr="sum"
        )

        self.conv2 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim,
                    aggr="mean"
                )
                for edge_type in edge_types
            },
            aggr="sum"
        )

        self.classifier = nn.Linear(
            hidden_dim,
            1
        )

    def forward(self, x_dict, edge_index_dict):

        x_dict = self.conv1(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            node_type: F.relu(x)
            for node_type, x in x_dict.items()
        }

        x_dict = {
            node_type: F.dropout(
                x,
                p=self.dropout,
                training=self.training
            )
            for node_type, x in x_dict.items()
        }

        x_dict = self.conv2(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            node_type: F.relu(x)
            for node_type, x in x_dict.items()
        }

        x_dict = {
            node_type: F.dropout(
                x,
                p=self.dropout,
                training=self.training
            )
            for node_type, x in x_dict.items()
        }

        customer_embedding = x_dict["customer"]

        logits = self.classifier(
            customer_embedding
        ).squeeze(-1)

        return logits, x_dict


model = HeteroGraphSAGE(
    metadata=data.metadata(),
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(DEVICE)

print(model)


# 9. Initialize Lazy GraphSAGE Layers

The `SAGEConv((-1, -1), ...)` configuration lets PyTorch Geometric infer the source and destination feature dimensions from the actual heterogeneous graph during the first forward pass.

Run one forward pass before creating the optimizer so all lazy parameters are initialized.


In [ ]:
# ============================================================
# 8. Initialize Lazy Parameters
# ============================================================

model.train()

with torch.no_grad():
    initial_logits, initial_embeddings = model(
        data.x_dict,
        data.edge_index_dict
    )

print("Initial forward pass completed.")
print("Customer logits shape:", initial_logits.shape)
print(
    "Customer embedding shape:",
    initial_embeddings["customer"].shape
)

assert initial_logits.shape[0] == num_customers


In [ ]:
# ============================================================
# 9. Model Parameter Summary
# ============================================================

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Total trainable parameters: {trainable_parameters:,}")


# 10. Loss and Optimizer

The initial experiment uses:

- `BCEWithLogitsLoss`
- training-derived positive-class weighting
- Adam optimizer
- weight decay

No hyperparameter search is performed here.


In [ ]:
# ============================================================
# 10. Loss and Optimizer
# ============================================================

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Loss:", criterion)
print("Optimizer:", optimizer)


# 11. Metric Helper

Metrics are calculated from probabilities using a fixed threshold of `0.5` during this training notebook.

This threshold is **not** the final deployment threshold.

Threshold optimization is explicitly reserved for the later threshold/error-analysis stage.


In [ ]:
# ============================================================
# 11. Metrics
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold=0.5
):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    metrics = {
        "accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(
            y_true,
            probabilities
        )
        metrics["pr_auc"] = average_precision_score(
            y_true,
            probabilities
        )
    else:
        metrics["roc_auc"] = float("nan")
        metrics["pr_auc"] = float("nan")

    return metrics


# 12. Evaluation Function

The model produces logits for every customer in the full graph.

The evaluation function selects only the requested customer mask.

Therefore:

```text
Full graph forward pass
        ↓
Customer logits
        ↓
Requested mask
        ↓
Metrics
```

Validation metrics never use test customers.


In [ ]:
# ============================================================
# 12. Evaluation Function
# ============================================================

@torch.no_grad()
def evaluate_mask(
    model,
    data,
    mask
):
    model.eval()

    logits, _ = model(
        data.x_dict,
        data.edge_index_dict
    )

    probabilities = torch.sigmoid(
        logits[mask]
    ).detach().cpu().numpy()

    targets = (
        data["customer"].y[mask]
        .detach()
        .cpu()
        .numpy()
    )

    loss = criterion(
        logits[mask],
        data["customer"].y[mask].float()
    ).item()

    metrics = calculate_metrics(
        targets,
        probabilities,
        threshold=0.5
    )

    metrics["loss"] = loss

    return metrics, probabilities, targets


# 13. Training Loop

Training policy:

- Maximum 150 epochs
- Early stopping patience = 20
- Best checkpoint selected using validation PR-AUC
- Training loss calculated only on `train_mask`
- Validation metrics calculated only on `val_mask`
- Test mask remains untouched

The checkpoint stores:

- model state
- optimizer state
- configuration
- best epoch
- validation metrics
- graph metadata required to reconstruct the model


In [ ]:
# ============================================================
# 13. Training Loop
# ============================================================

history = []

best_metric = -np.inf
best_epoch = -1
epochs_without_improvement = 0

best_state = None
best_optimizer_state = None
best_val_metrics = None

for epoch in range(1, MAX_EPOCHS + 1):

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.train()
    optimizer.zero_grad(set_to_none=True)

    logits, _ = model(
        data.x_dict,
        data.edge_index_dict
    )

    train_loss = criterion(
        logits[train_mask],
        data["customer"].y[train_mask].float()
    )

    train_loss.backward()

    optimizer.step()

    # --------------------------------------------------------
    # Training metrics
    # --------------------------------------------------------

    train_metrics, _, _ = evaluate_mask(
        model,
        data,
        train_mask
    )

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------

    val_metrics, _, _ = evaluate_mask(
        model,
        data,
        val_mask
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": float(train_loss.item()),
        "val_loss": float(val_metrics["loss"]),
        "train_accuracy": float(train_metrics["accuracy"]),
        "train_precision": float(train_metrics["precision"]),
        "train_recall": float(train_metrics["recall"]),
        "train_f1": float(train_metrics["f1"]),
        "train_roc_auc": float(train_metrics["roc_auc"]),
        "train_pr_auc": float(train_metrics["pr_auc"]),
        "val_accuracy": float(val_metrics["accuracy"]),
        "val_precision": float(val_metrics["precision"]),
        "val_recall": float(val_metrics["recall"]),
        "val_f1": float(val_metrics["f1"]),
        "val_roc_auc": float(val_metrics["roc_auc"]),
        "val_pr_auc": float(val_metrics["pr_auc"]),
    }

    history.append(epoch_record)

    current_metric = val_metrics["pr_auc"]

    improved = (
        np.isfinite(current_metric)
        and current_metric > best_metric
    )

    if improved:

        best_metric = current_metric
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state = copy.deepcopy(
            model.state_dict()
        )

        best_optimizer_state = copy.deepcopy(
            optimizer.state_dict()
        )

        best_val_metrics = dict(
            val_metrics
        )

    else:
        epochs_without_improvement += 1

    if (
        epoch == 1
        or epoch % 5 == 0
        or improved
    ):
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss {train_loss.item():.4f} | "
            f"Val Loss {val_metrics['loss']:.4f} | "
            f"Val F1 {val_metrics['f1']:.4f} | "
            f"Val ROC-AUC {val_metrics['roc_auc']:.4f} | "
            f"Val PR-AUC {val_metrics['pr_auc']:.4f}"
            + ("  ← best" if improved else "")
        )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"Early stopping at epoch {epoch}. "
            f"Best epoch: {best_epoch}"
        )
        break

if best_state is None:
    raise RuntimeError(
        "No valid validation checkpoint was produced."
    )

print("\nTraining completed.")
print("Best epoch:", best_epoch)
print("Best validation PR-AUC:", best_metric)


# 14. Restore Best Validation Checkpoint

The model is restored to the epoch with the best validation PR-AUC.

This is **not yet the final model**.

It is simply the best checkpoint from this initial training experiment.


In [ ]:
# ============================================================
# 14. Restore Best State
# ============================================================

model.load_state_dict(best_state)

restored_val_metrics, val_probabilities, val_targets = evaluate_mask(
    model,
    data,
    val_mask
)

print("Best checkpoint restored.")
print("\nValidation metrics:")
display(
    pd.DataFrame([restored_val_metrics]).T.rename(
        columns={0: "value"}
    )
)


# 15. Save Best Checkpoint

The checkpoint is saved for later inspection and tuning.

It is deliberately named:

```text
graphsage_best_checkpoint.pt
```

rather than `final_model.pt`.

This distinction matters because the model has **not** been accepted/frozen yet.


In [ ]:
# ============================================================
# 15. Save Best Checkpoint
# ============================================================

model_config = {
    "model_class": "HeteroGraphSAGE",
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "primary_metric": PRIMARY_METRIC,
    "threshold_for_training_metrics": 0.5,
    "pos_weight": float(pos_weight_value),
    "node_types": list(data.node_types),
    "edge_types": [
        list(edge_type)
        for edge_type in data.edge_types
    ],
    "customer_feature_dimension": int(
        data["customer"].x.shape[1]
    ),
}

checkpoint = {
    "model_state_dict": best_state,
    "optimizer_state_dict": best_optimizer_state,
    "model_config": model_config,
    "best_epoch": int(best_epoch),
    "best_validation_metrics": {
        key: float(value)
        for key, value in restored_val_metrics.items()
    },
    "graph_metadata": graph_metadata,
}

torch.save(
    checkpoint,
    CHECKPOINT_PATH
)

assert CHECKPOINT_PATH.exists()

print(f"Best checkpoint saved to: {CHECKPOINT_PATH}")
print(
    f"Checkpoint size: "
    f"{CHECKPOINT_PATH.stat().st_size / (1024 ** 2):.2f} MB"
)


# 16. Save Training History

The complete epoch-by-epoch history is saved as JSON.

This makes the experiment reproducible and allows later comparison with tuning runs.


In [ ]:
# ============================================================
# 16. Save Training History
# ============================================================

history_payload = {
    "configuration": model_config,
    "best_epoch": int(best_epoch),
    "best_validation_metrics": {
        key: float(value)
        for key, value in restored_val_metrics.items()
    },
    "history": history,
}

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history_payload, f, indent=2)

assert HISTORY_PATH.exists()

print(f"Training history saved to: {HISTORY_PATH}")


# 17. Training Curves

Visualize:

1. Training vs validation loss
2. Validation F1
3. Validation ROC-AUC
4. Validation PR-AUC

These curves are diagnostic evidence for:

- underfitting
- overfitting
- convergence
- instability

They are not used to tune the model automatically in this notebook.


In [ ]:
# ============================================================
# 17. Plot Training Curves
# ============================================================

history_df = pd.DataFrame(history)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10)
)

axes = axes.flatten()

# Loss
axes[0].plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train Loss"
)
axes[0].plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Validation Loss"
)
axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# F1
axes[1].plot(
    history_df["epoch"],
    history_df["val_f1"],
    label="Validation F1"
)
axes[1].axvline(
    best_epoch,
    linestyle="--",
    label="Best Epoch"
)
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")
axes[1].legend()

# ROC-AUC
axes[2].plot(
    history_df["epoch"],
    history_df["val_roc_auc"],
    label="Validation ROC-AUC"
)
axes[2].axvline(
    best_epoch,
    linestyle="--",
    label="Best Epoch"
)
axes[2].set_title("Validation ROC-AUC")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("ROC-AUC")
axes[2].legend()

# PR-AUC
axes[3].plot(
    history_df["epoch"],
    history_df["val_pr_auc"],
    label="Validation PR-AUC"
)
axes[3].axvline(
    best_epoch,
    linestyle="--",
    label="Best Epoch"
)
axes[3].set_title("Validation PR-AUC")
axes[3].set_xlabel("Epoch")
axes[3].set_ylabel("PR-AUC")
axes[3].legend()

plt.tight_layout()

plt.savefig(
    CURVE_PATH,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(f"Training curves saved to: {CURVE_PATH}")


# 18. Best Validation Results

These are the results from the **initial GraphSAGE configuration**.

They are validation results only.

The test set has not been evaluated.


In [ ]:
# ============================================================
# 18. Best Validation Results
# ============================================================

validation_results = pd.DataFrame([{
    "best_epoch": best_epoch,
    "loss": restored_val_metrics["loss"],
    "accuracy": restored_val_metrics["accuracy"],
    "precision": restored_val_metrics["precision"],
    "recall": restored_val_metrics["recall"],
    "f1": restored_val_metrics["f1"],
    "roc_auc": restored_val_metrics["roc_auc"],
    "pr_auc": restored_val_metrics["pr_auc"],
}])

display(validation_results.T.rename(columns={0: "value"}))

print("Primary selection metric: Validation PR-AUC")
print(f"Best validation PR-AUC: {restored_val_metrics['pr_auc']:.6f}")


# 19. Validation Confusion Matrix

This is a validation-only diagnostic using the temporary 0.5 threshold.

The threshold is **not frozen** here.


In [ ]:
# ============================================================
# 19. Validation Confusion Matrix
# ============================================================

val_predictions = (
    val_probabilities >= 0.5
).astype(int)

cm = confusion_matrix(
    val_targets,
    val_predictions
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(cm_df)


# 20. Verify the Test Set Has Not Been Used for Model Selection

Notebook 05 intentionally does not calculate test predictions.

The test mask is present because it is part of the graph contract, but it is not passed to the evaluation function during training or checkpoint selection.

The final test evaluation belongs to the post-freeze stage.


In [ ]:
# ============================================================
# 20. Test-Set Protection Check
# ============================================================

# Explicitly confirm that no test predictions were produced.
test_predictions_created = False

assert test_predictions_created is False

print("Test-set protection check passed.")
print("No test predictions were generated for model selection.")


# 21. Final Artifact Verification

Required Notebook 05 artifacts:

```text
artifacts/
├── models/
│   └── graphsage_best_checkpoint.pt
│
└── results/
    ├── graphsage_training_history.json
    └── graphsage_training_curves.png
```

The checkpoint is a **candidate model**, not the frozen production model.


In [ ]:
# ============================================================
# 21. Artifact Verification
# ============================================================

required_artifacts = [
    CHECKPOINT_PATH,
    HISTORY_PATH,
    CURVE_PATH,
]

artifact_rows = []

for path in required_artifacts:
    artifact_rows.append({
        "artifact": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    })

artifact_df = pd.DataFrame(artifact_rows)

display(artifact_df)

assert artifact_df["exists"].all()

print("All required Notebook 05 artifacts exist.")


# 22. Reload Checkpoint

Before finishing, reload the saved checkpoint and verify that its configuration and state can be restored.

This does not evaluate the test set.


In [ ]:
# ============================================================
# 22. Reload Checkpoint
# ============================================================

try:
    loaded_checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False
    )
except TypeError:
    loaded_checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE
    )

required_checkpoint_keys = {
    "model_state_dict",
    "optimizer_state_dict",
    "model_config",
    "best_epoch",
    "best_validation_metrics",
    "graph_metadata",
}

assert required_checkpoint_keys.issubset(
    loaded_checkpoint.keys()
)

print("Checkpoint reload passed.")
print("Saved best epoch:", loaded_checkpoint["best_epoch"])
print(
    "Saved validation PR-AUC:",
    loaded_checkpoint["best_validation_metrics"]["pr_auc"]
)


# 23. Final Automated Verification

Notebook 05 is complete only when:

- [x] Saved graph loads successfully.
- [x] Customer features/labels/masks are valid.
- [x] Model is heterogeneous GraphSAGE.
- [x] Loss uses training-derived class weighting.
- [x] Training uses `train_mask`.
- [x] Validation uses `val_mask`.
- [x] Test data is not used for checkpoint selection.
- [x] Early stopping is enabled.
- [x] Best checkpoint is selected using validation PR-AUC.
- [x] Training history is saved.
- [x] Training curves are saved.
- [x] Best checkpoint is reloadable.
- [x] No final test evaluation is performed.

## Important Stop Point

**STOP HERE.**

Do not:

- tune hyperparameters
- evaluate the test set
- freeze the final model
- build the backend
- build the frontend

The next decision belongs to the user.

Inspect the validation results and training curves.

If performance is not satisfactory, the next step should be a controlled optimization experiment.

If performance is satisfactory, the user should explicitly say:

```text
MODEL IS SATISFACTORY
```

Only then should the final model-freezing stage begin.


In [ ]:
# ============================================================
# 23. FINAL AUTOMATED VERIFICATION
# ============================================================

assert isinstance(data, HeteroData)
assert CHECKPOINT_PATH.exists()
assert HISTORY_PATH.exists()
assert CURVE_PATH.exists()

assert best_epoch >= 1
assert len(history) >= best_epoch

assert restored_val_metrics["pr_auc"] >= 0.0
assert restored_val_metrics["pr_auc"] <= 1.0

assert restored_val_metrics["roc_auc"] >= 0.0
assert restored_val_metrics["roc_auc"] <= 1.0

assert test_predictions_created is False

with open(HISTORY_PATH, "r", encoding="utf-8") as f:
    saved_history = json.load(f)

assert saved_history["best_epoch"] == int(best_epoch)
assert len(saved_history["history"]) == len(history)

print("=" * 85)
print("NOTEBOOK 05 VERIFICATION PASSED")
print("=" * 85)
print("Model               : Heterogeneous GraphSAGE")
print("Device              :", DEVICE)
print("Hidden dimension    :", HIDDEN_DIM)
print("Dropout             :", DROPOUT)
print("Learning rate       :", LEARNING_RATE)
print("Weight decay        :", WEIGHT_DECAY)
print("Best epoch          :", best_epoch)
print("Validation PR-AUC   :", f"{restored_val_metrics['pr_auc']:.6f}")
print("Validation ROC-AUC  :", f"{restored_val_metrics['roc_auc']:.6f}")
print("Validation F1       :", f"{restored_val_metrics['f1']:.6f}")
print("Validation Precision:", f"{restored_val_metrics['precision']:.6f}")
print("Validation Recall   :", f"{restored_val_metrics['recall']:.6f}")
print("Checkpoint          :", CHECKPOINT_PATH)
print("History             :", HISTORY_PATH)
print("Curves              :", CURVE_PATH)
print("=" * 85)
print("STOP: Inspect validation performance before any tuning or final evaluation.")
